# Amazon Reviews Exploratory Data Analysis

This notebook is for dataset inspection only. Use it before running the production Spark training and streaming pipeline to check raw columns, sentiment label distribution, and the deterministic train/validation/test split.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path('../data/raw/Reviews.csv')
RANDOM_SEED = 42
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

pd.set_option('display.max_colwidth', 140)
DATA_PATH

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing dataset: {DATA_PATH.resolve()}')

df = pd.read_csv(DATA_PATH)
df.shape

In [ ]:
df.head()

In [ ]:
important_columns = ['ProductId', 'UserId', 'Score', 'Time', 'Summary', 'Text']
df[important_columns].isna().sum().to_frame('missing_count')

In [ ]:
eda_df = df[important_columns].dropna(subset=['Score', 'Text']).copy()
eda_df = eda_df[eda_df['Score'].between(1, 5)]

def sentiment_label(score):
    if score < 3:
        return 'negative'
    if score == 3:
        return 'neutral'
    return 'positive'

eda_df['label'] = eda_df['Score'].map(sentiment_label)
eda_df['review_date'] = pd.to_datetime(eda_df['Time'], unit='s', errors='coerce').dt.date
eda_df[['Score', 'label', 'review_date', 'Text']].head()

In [ ]:
label_distribution = (
    eda_df['label']
    .value_counts()
    .rename_axis('label')
    .reset_index(name='count')
)
label_distribution['percent'] = label_distribution['count'] / label_distribution['count'].sum() * 100
label_distribution

In [ ]:
split_df = eda_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
train_end = int(len(split_df) * TRAIN_RATIO)
val_end = int(len(split_df) * (TRAIN_RATIO + VAL_RATIO))

split_df['split'] = 'test'
split_df.loc[:train_end - 1, 'split'] = 'train'
split_df.loc[train_end:val_end - 1, 'split'] = 'validation'

split_counts = pd.crosstab(split_df['split'], split_df['label'], margins=True)
split_counts

In [ ]:
split_percent = pd.crosstab(split_df['split'], split_df['label'], normalize='index') * 100
split_percent.round(2)

In [ ]:
split_df['text_length'] = split_df['Text'].str.len()
split_df.groupby('split')['text_length'].describe().round(2)